# Vextor – training su Colab

Addestra il tuo modello per siti web (HTML/CSS/JS) partendo da **Qwen2.5-Coder** e lo esporta in **GGUF** per l'app Android.

1. `Runtime → Cambia tipo di runtime → GPU` (T4 gratuita va bene per il modello 1.5B).
2. Esegui le celle in ordine.
3. Alla fine scarica il file `.gguf` (o caricalo su Hugging Face) e importalo nell'app Vextor.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Codice
Se il repository è privato, inserisci un GitHub token con accesso in lettura quando richiesto (lascia vuoto se è pubblico).

In [ ]:
import getpass, os
REPO = "TiclyMusic/vextor1.0"
BRANCH = "main"  # cambia se lavori su un altro branch
token = getpass.getpass("GitHub token (vuoto se il repo è pubblico): ").strip()
auth = f"{token}@" if token else ""
!rm -rf vextor && git clone -q --depth 1 -b {BRANCH} https://{auth}github.com/{REPO} vextor
%cd vextor
!pip install -q -r training/requirements.txt

## 2. Dataset
**Opzione A (consigliata):** scarica il dataset prodotto dalla GitHub Action *Dataset (scraping GitHub)*.

**Opzione B:** esegui lo scraping qui (serve un GitHub token, ~1 ora per 1200 repo).

In [ ]:
USE_RELEASE = True  # False = opzione B
import urllib.request, urllib.error, tarfile, zipfile, json, io

def gh_get(url, accept="application/vnd.github+json"):
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {token}", "Accept": accept})
    return urllib.request.urlopen(req).read()

def download_dataset():
    # 1) release "dataset-latest"
    try:
        if token:
            rel = json.loads(gh_get(f"https://api.github.com/repos/{REPO}/releases/tags/dataset-latest"))
            asset = [a for a in rel["assets"] if a["name"] == "dataset.tar.gz"][0]
            data = gh_get(asset["url"], "application/octet-stream")
        else:
            url = f"https://github.com/{REPO}/releases/download/dataset-latest/dataset.tar.gz"
            data = urllib.request.urlopen(url).read()
        return data
    except (urllib.error.HTTPError, IndexError) as e:
        print("Release non disponibile:", e)
    # 2) ultimo artifact "dataset" della GitHub Action (serve il token)
    if not token:
        raise SystemExit("Inserisci un GitHub token nella prima cella per scaricare l'artifact.")
    arts = json.loads(gh_get(f"https://api.github.com/repos/{REPO}/actions/artifacts?name=dataset&per_page=1"))["artifacts"]
    z = zipfile.ZipFile(io.BytesIO(gh_get(arts[0]["archive_download_url"])))
    return z.read("dataset.tar.gz")

if USE_RELEASE:
    open("dataset.tar.gz", "wb").write(download_dataset())
    os.makedirs("data", exist_ok=True)
    tarfile.open("dataset.tar.gz").extractall("data")
else:
    os.environ["GITHUB_TOKEN"] = token or getpass.getpass("GitHub token per lo scraping: ")
    !python scraper/scrape_repos.py --max-repos 4000 --out data/raw
    !python scraper/build_dataset.py --raw data/raw --out data/dataset
!cat data/dataset/stats.json
!wc -l data/dataset/*.jsonl

## 3. Training LoRA
- **T4 (16 GB):** `BASE=1.5B`, `MAX_LEN=6144`, `EPOCHS=1` (~1-2 ore)
- **L4 / A100:** puoi usare `MAX_LEN=8192`, `EPOCHS=2` o il modello 3B

Se Colab si disconnette, riesegui la cella: riparte dall'ultimo checkpoint (`--resume`).

In [ ]:
BASE = "Qwen/Qwen2.5-Coder-1.5B-Instruct"   # oppure "Qwen/Qwen2.5-Coder-3B-Instruct"
MAX_LEN = 6144
EPOCHS = 1
!python training/train_lora.py --base {BASE} --data data/dataset --out out/vextor-lora \
    --max-len {MAX_LEN} --epochs {EPOCHS} --resume

## 4. Prova veloce

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from IPython.display import HTML, display
import re, sys
sys.path.insert(0, "scraper")
from build_dataset import SYSTEM_PROMPT

tok = AutoTokenizer.from_pretrained("out/vextor-lora")
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map="cuda")
model = PeftModel.from_pretrained(model, "out/vextor-lora").eval()
msgs = [{"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Crea una landing page per un'app di meditazione, tema scuro, palette viola."}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
out = model.generate(**ids, max_new_tokens=3000, temperature=0.3, top_p=0.9, do_sample=True)
answer = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
html = (re.search(r"```html\n(.*?)(```|$)", answer, re.S) or [None, answer])[1]
display(HTML(f'<iframe srcdoc="{html.replace(chr(34), "&quot;")}" style="width:100%;height:600px;border:1px solid #ccc"></iframe>'))
del model; torch.cuda.empty_cache()

## 5. Unione LoRA ed export GGUF
`q8_0` (~1.6 GB per 1.5B) è il default: ottima qualità e gira bene sul Galaxy S25.
Per un file più piccolo usa `q4_k_m` (compila llama.cpp, qualche minuto in più).

In [ ]:
QUANT = "q8_0"
!python training/train_lora.py --merge --out out/vextor-lora --merged out/vextor-merged
!bash training/export_gguf.sh out/vextor-merged out/vextor-{QUANT}.gguf {QUANT}

## 6. Porta il modello sul telefono
Scegli **una** delle opzioni:
- **Hugging Face** (consigliato): carica il file e incolla il link nell'app → *Modelli → Scarica dal link*.
- **Google Drive**: copia su Drive, poi dal telefono scaricalo e importalo con *Modelli → Scegli file .gguf*.

In [ ]:
DEST = "huggingface"   # "huggingface" | "drive" | "download"
GGUF = f"out/vextor-{QUANT}.gguf"
if DEST == "huggingface":
    from huggingface_hub import HfApi, login
    login()  # token con permesso di scrittura
    api = HfApi()
    repo_id = api.whoami()["name"] + "/vextor-gguf"
    api.create_repo(repo_id, exist_ok=True)
    api.upload_file(path_or_fileobj=GGUF, path_in_repo=os.path.basename(GGUF), repo_id=repo_id)
    print("Link da incollare nell'app:")
    print(f"https://huggingface.co/{repo_id}/resolve/main/{os.path.basename(GGUF)}")
elif DEST == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    !cp {GGUF} /content/drive/MyDrive/
    print("Copiato su Google Drive:", os.path.basename(GGUF))
else:
    from google.colab import files
    files.download(GGUF)